In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report

# 1. Generate Synthetic Agricultural Telemetry Dataset
np.random.seed(42)
num_samples = 500

transit_hours = np.random.uniform(6, 96, num_samples)          # 6 to 96 hours
temperature = np.random.uniform(18, 42, num_samples)           # 18°C to 42°C
humidity = np.random.uniform(40, 98, num_samples)              # 40% to 98%

# Spoilage rule logic (High heat + high humidity + long transit = high risk)
spoilage_risk = [
    1 if (t > 30.0 and h > 75.0) or (hr > 48.0 and t > 28.0) else 0
    for hr, t, h in zip(transit_hours, temperature, humidity)
]

df = pd.DataFrame({
    'transit_hours': transit_hours,
    'temperature': temperature,
    'humidity': humidity,
    'spoilage_risk': spoilage_risk
})

# 2. Train-Test Split
X = df[['transit_hours', 'temperature', 'humidity']]
y = df['spoilage_risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Model Training
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred))

# 4. Export Decision Rules for Lightweight n8n Python Node Deployment
rules = export_text(model, feature_names=['transit_hours', 'temperature', 'humidity'])
print("=== EXTRACTED DECISION TREE RULES FOR N8N ===")
print(rules)

Model Accuracy: 96.00%

              precision    recall  f1-score   support

           0       0.97      0.97      0.97        61
           1       0.95      0.95      0.95        39

    accuracy                           0.96       100
   macro avg       0.96      0.96      0.96       100
weighted avg       0.96      0.96      0.96       100

=== EXTRACTED DECISION TREE RULES FOR N8N ===
|--- temperature <= 28.21
|   |--- class: 0
|--- temperature >  28.21
|   |--- transit_hours <= 48.29
|   |   |--- humidity <= 75.49
|   |   |   |--- class: 0
|   |   |--- humidity >  75.49
|   |   |   |--- class: 1
|   |--- transit_hours >  48.29
|   |   |--- class: 1

